1- Coleta dos dados <br>

*   Dataset de câncer de mama Wisconsin
*   Problema binário: Diagnóstico do exame, podendo ser **maligno** ou **benigno**
*   Abordagem: Aprendizado supervisionado

2- Preparação dos dados <br>


*   Dataset escolhido: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data/data
*   Nome do arquivo: data.csv
*   Bibliotecas importadas: pandas

3- Escolha do modelo (supervisionado) <br>

*   Regressão Logística
*   Random Forest

4- Treinar o modelo <br>
5- Testes <br>
6- Deploy (Opcional) <br>
7- Visão computacional (Opcional)

#Importação dos dados

In [ ]:
import pandas as pd
from google.colab import files, data_table
import io
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Realiza o download do arquivo salvo no drive
!gdown --id "1z3JF1r6ixWJsp_C6rDeWCHXP1kUIxQtD"

#Exibição dos dados

In [ ]:
# Nome do arquivo
file_name = 'data.csv'

# Carrega o CSV para um DataFrame
df = pd.read_csv(f"./{file_name}")

df.head()

In [ ]:
# Colunas dropadas
df.drop(['Unnamed: 32', 'id'], axis=1, inplace=True)

print(f"Arquivo '{file_name}' carregado com sucesso!")

# Exibe explicitamente o DataTable com 10 linhas por página e suporte a todas as colunas
display(data_table.DataTable(df, include_index=False, num_rows_per_page=15, max_columns=40))

# Análise Gráfica Malígno X Benigno

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='diagnosis', data=df)
plt.title('Distribuição da Coluna Diagnóstico')
plt.xlabel('Diagnóstico')
plt.ylabel('Contagem')
plt.show()

# **Análise inicial (Eric):**

In [ ]:
# Primeiras linhas do dataset
df.info()

# Análise dos dados (Eric)

In [ ]:
# Descrições estatísticas
df.describe()

In [ ]:
# Matriz de correlação

df["diagnostico_numero"] = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

correlation = df.corr(numeric_only=True)

plt.figure(figsize=(12, 10))

sns.heatmap(
    df.corr(numeric_only=True),
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".1f"
)

plt.title("Correlação Matrix")
plt.show()

In [ ]:
all_numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
# Remove 'id' e 'diagnostico_numero' da lista de features para evitar correlação com si mesmas ou ID
features_to_correlate = [col for col in all_numeric_cols if col not in ['id', 'diagnostico_numero','diagnosis_numeric']]

# Adiciona 'diagnostico_numero' para calcular a correlação com as outras features
selected_features_for_corr = features_to_correlate + ['diagnostico_numero']

# Calcula a matriz de correlação para as características selecionadas
correlation_subset = df[selected_features_for_corr].corr()

# Exibe a correlação dessas características com 'diagnostico_numero'
display(correlation_subset['diagnostico_numero'].sort_values(ascending=False))

#Algoritmo de Regressão Logística

In [ ]:
# Treino
X = df.drop(["diagnosis", "diagnostico_numero"], axis=1)

# Target
y = df["diagnostico_numero"]

In [ ]:
pipeline_cv_full = Pipeline([
    ('scaler', StandardScaler()), # Padronização
    ('classifier', LogisticRegression(random_state=42, solver='liblinear')) # Classificação
])

# Configuração da validação cruzada
skf_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Validação com métrica de acurácia
scores_full_cv = cross_val_score(pipeline_cv_full, X, y, cv=skf_full, scoring='accuracy')

print("Acurácias para cada fold da validação cruzada")
for i, score in enumerate(scores_full_cv):
    print(f"Fold {i+1}: {score * 100:.2f}%")
print("---------------------------------------------------")
print(f"Acurácia média da validação cruzada: {scores_full_cv.mean() * 100:.2f}%") # Média entre os Folds
print(f"Desvio padrão das acurácias: {scores_full_cv.std() * 100:.2f}%") # Devio padrão

# Random Forest

In [ ]:
# 1. Separação das Variáveis
x = df.drop(["diagnosis", "diagnostico_numero"], axis=1)
y = df['diagnosis']

In [ ]:
# 2. Divisão dos Dados (70% Treino, 30% Teste)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, stratify=y, random_state=42)

# 3. Definição do Modelo e Grade de Hiperparâmetros
rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [10, 30, 50, 70, 90], # Quantidade de árvores normais para o modelo
    'max_depth': [2, 4, 6, 8, 10]        # Testando diferentes profundidades
}

In [ ]:
# 4. Busca pelos Melhores Hiperparâmetros
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

grid_search.fit(x_train, y_train)

# 5. Resultados
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")

In [ ]:
melhor_modelo = grid_search.best_estimator_

y_predict_rf = melhor_modelo.predict(x_test)
print(f"Acurácia final no Teste: {accuracy_score(y_test, y_predict_rf):.4f}")

### **Tarefa extra - Visão Computacional**

**1. Coleta dos dados**

Dataset CBIS-DDSM (mamografias), Kaggle

Problema binário: maligno ou benigno

Abordagem: aprendizado supervisionado

**2. Preparação dos dados**

kaggle.com/datasets/awsaf49/cbis-ddsm

Arquivos: mass_case_description, dicom_info.csv

Imagens redimensionadas 224x224, normalizadas

Bibliotecas: pandas, TensorFlow, scikit-learn

**3. Escolha do modelo (supervisionado)**

CNN simples, construída do zero

3 blocos convolucionais (32, 64, 128 filtros

**4. Treinar o modelo**

10 épocas, otimizador adam

922 imagens de treino / 198 validação

**5. Testes**

198 imagens nunca vistas pelo modelo

Métricas: acurácia, recall, F1-score

Recall da classe maligno como métrica central

Matriz de confusão e classification_report

Instalação da biblioteca do Kaggle com envio da credencial de API e download do dataset CBIS-DDSM.

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset
!unzip -q cbis-ddsm-breast-cancer-image-dataset.zip -d cbis_ddsm

Carregado o arquivo de descrição dos casos de massas mamárias, que traz, para cada exame, informações como densidade da mama, lateralidade, e o diagnóstico (pathology)

In [ ]:
import pandas as pd
import os

df = pd.read_csv(
    "/content/cbis_ddsm/csv/mass_case_description_train_set.csv"
)

print(df.head())
print(df.columns)

In [ ]:
Conversão da coluna de diagnóstico em um rótulo binário: 0 para benigno, 1 para maligno.

In [ ]:
df["label"] = df["pathology"].map({
    "BENIGN": 0,
    "BENIGN_WITHOUT_CALLBACK": 0,
    "MALIGNANT": 1
})

Divisçao dos dados em 70% treino, 15% validação e 15% teste, usando estratificação para manter a mesma proporção de casos malignos e benignos em cada grupo. Isso resultou em 922 imagens de treino, 198 de validação e 198 de teste.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

In [ ]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

In [ ]:
print(df.columns.tolist())

Inicialmente, tentamos montar os caminhos das imagens diretamente a partir do CSV. Identificamos, porém, que esses caminhos apontavam para arquivos .dcm, formato DICOM original, que não correspondiam à estrutura real de arquivos .jpg disponível no dataset baixado.

In [ ]:
BASE_DIR = "/content/cbis_ddsm"

train_paths = [
    os.path.join(BASE_DIR, p)
    for p in train_df["image file path"]
]

val_paths = [
    os.path.join(BASE_DIR, p)
    for p in val_df["image file path"]
]

test_paths = [
    os.path.join(BASE_DIR, p)
    for p in test_df["image file path"]
]

Definição do tamanho padrão das imagens (224x224 pixels) e o tamanho do lote de treino. A função process_image lê cada arquivo, decodifica como JPEG, redimensiona e normaliza os valores de pixel para o intervalo 0-1

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
def process_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = image / 255.0

    return image, label